# Phase 3 — Standard LSTM: Colab Training

The single Phase 3 notebook runs the **same Python modules** used by the local project. It trains one message-level LSTM per task. It never passes other messages, speaker history, power scores or game context into the model.

**Run-time:** Google Colab Free Tier (GPU when available, CPU fallback). **Storage:** Google Drive; generated files remain in `artifacts/`. The original three files in `data/` are not modified.

**Original configuration:** bidirectional 100-hidden LSTM, 200d frozen Twitter GloVe, max pooling, dropout .5, Adam .003, weighted loss, 15 epochs and validation-based checkpointing. Some Colab safeguards and tokenizer behavior differ from the original AllenNLP code; see `docs/phase_3_standard_lstm.md`.

**Training has not been performed in Colab as part of this deliverable.** All results shown after training will be generated during your own execution.

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
import zipfile

IN_COLAB = Path('/content').exists() and 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
print('Colab:', IN_COLAB)

## Mount Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_HOME = Path('/content/drive/MyDrive/diplomacy_lstm')
else:
    PROJECT_HOME = Path.cwd()
PROJECT_HOME.mkdir(parents=True, exist_ok=True)
print('Project home:', PROJECT_HOME)

## Obtain project code: ZIP or GitHub

Upload the **complete Phase 3 ZIP** under `/content/diplomacy-deception-detection-phase3.zip`, or set `GITHUB_URL` to your own GitHub repository. Leave an existing project directory in Drive untouched to resume a previous session.

In [ ]:
GITHUB_URL = ''  # Optional: your actual repository URL
PROJECT_ZIP = Path('/content/diplomacy-deception-detection-phase3.zip')

if (PROJECT_HOME / 'diplomacy-deception-detection' / 'src').is_dir():
    ROOT = PROJECT_HOME / 'diplomacy-deception-detection'
elif (PROJECT_HOME / 'src').is_dir():
    ROOT = PROJECT_HOME
elif GITHUB_URL:
    ROOT = PROJECT_HOME / 'diplomacy-deception-detection'
    subprocess.run(['git', 'clone', GITHUB_URL, str(ROOT)], check=True)
elif IN_COLAB and PROJECT_ZIP.is_file():
    with zipfile.ZipFile(PROJECT_ZIP) as archive:
        for item in archive.infolist():
            target = (PROJECT_HOME / item.filename).resolve()
            if not target.is_relative_to(PROJECT_HOME.resolve()):
                raise ValueError('Unsafe ZIP member path')
        archive.extractall(PROJECT_HOME)
    ROOT = PROJECT_HOME / 'diplomacy-deception-detection'
elif not IN_COLAB:
    ROOT = Path.cwd()
else:
    raise FileNotFoundError('Upload the complete Phase 3 ZIP under /content or set GITHUB_URL')

assert (ROOT / 'src' / 'diplomacy').is_dir(), f'Project root not found: {ROOT}'
print('Project root:', ROOT)

## Install missing dependencies

In [ ]:
import importlib.util
packages = {'numpy':'numpy', 'pandas':'pandas', 'sklearn':'scikit-learn',
            'matplotlib':'matplotlib', 'torch':'torch'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
print('Missing packages installed:', missing)

## Detect GPU or CPU

In [ ]:
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Verify original project data

In [ ]:
original_files = sorted(p.name for p in (ROOT / 'data').iterdir() if p.is_file())
assert original_files == ['test.jsonl', 'train.jsonl', 'validation.jsonl'], original_files
print('Original data:', original_files)

import hashlib
def data_hashes():
    return {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in (ROOT / 'data').glob('*.jsonl')}
RAW_HASHES_BEFORE = data_hashes()
print('Original file integrity hashes recorded.')

## Verify processed data

In [ ]:
for task in ['actual_lie', 'suspected_lie']:
    for split in ['train', 'validation', 'test']:
        file = ROOT / 'artifacts' / 'processed' / task / f'{split}.jsonl'
        print(task, split, ':', file.exists())
        assert file.is_file(), f'Missing: {file}' 

## Import the shared project modules

In [ ]:
sys.path.insert(0, str(ROOT / 'src'))
from diplomacy.data.text_dataset import Vocabulary, MessageDataset, tokenize
from diplomacy.data.text_collator import collate_messages
from diplomacy.models.neural.lstm import StandardLSTM
from diplomacy.training.standard_lstm_runner import (
    read_config, prepare, run_training, run_evaluation, compare_phase2,
    save_parity_reference, write_error_analysis,
)
from diplomacy.inference.standard_lstm import load_predictor
print('Imported production modules from:', ROOT / 'src')

## Load shared experiment configurations

In [ ]:
CONFIGS = {task: read_config(ROOT, task) for task in ['actual_lie', 'suspected_lie']}
for task, cfg in CONFIGS.items():
    print('\n', task)
    print('Architecture:', cfg['model'])
    print('Embedding:', cfg['embeddings'])
    print('Training:', cfg['training'])

## Training controls

Use these variables to run or resume either task independently. Downloading the original Twitter GloVe archive requires roughly 1.4 GB plus extraction/processing overhead; if it is already in `artifacts/embeddings/glove/`, no download is needed. An opt-in random embedding alternative requires editing the **shared YAML** first and is not a paper reproduction.

In [ ]:
RUN_ACTUAL = True
RUN_SUSPECTED = True
RESUME = True
DOWNLOAD_GLOVE = True
RUN_TEST_EVALUATION = True
print('Task selection:', RUN_ACTUAL, RUN_SUSPECTED)
print('GloVe download if missing:', DOWNLOAD_GLOVE)

## Vocabulary construction

Vocabularies are fitted separately on the training data only, inside the shared `prepare()` function. Validation and test messages cannot change their token IDs.

In [ ]:
actual_prepared = None
if RUN_ACTUAL:
    actual_prepared = prepare(ROOT, 'actual_lie', device=DEVICE,
                              download_glove=DOWNLOAD_GLOVE)
    print('Actual-lie vocabulary:', len(actual_prepared['vocab']))
    print('First tokens:', actual_prepared['vocab'].id_to_token[:15])

## Embedding initialization

The preparation step above initializes the original pretrained GloVe embeddings (or reloads an existing checkpoint). It records embedding coverage and provenance.

In [ ]:
if actual_prepared is not None:
    print('Embedding information:', actual_prepared['embedding_info'])
    print('Embedding weight shape:', tuple(actual_prepared['model'].embedding.weight.shape))

## Dataset and DataLoader inspection

In [ ]:
if actual_prepared is not None:
    batch = next(iter(actual_prepared['loaders']['train']))
    print('Token IDs:', tuple(batch['input_ids'].shape))
    print('Sequence lengths:', tuple(batch['lengths'].shape))
    print('Targets:', tuple(batch['targets'].shape))
    print('First message ID:', batch['message_ids'][0])

## Standard LSTM architecture

Token IDs → frozen 200d embeddings → bidirectional LSTM (100 dimensions per direction) → **masked max pooling** → dropout → two logits. Padding cannot contribute to the pooled representation.

## Model initialization and forward tensor shapes

In [ ]:
if actual_prepared is not None:
    model = actual_prepared['model'].to(DEVICE)
    model.eval()
    with torch.inference_mode():
        logits = model(batch['input_ids'].to(DEVICE), batch['lengths'].to(DEVICE))
    print('Embedding dimension:', model.embedding.embedding_dim)
    print('LSTM output dimension:', model.classifier.in_features)
    print('Logit tensor:', tuple(logits.shape))
    model.cpu()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## Actual-lie training

The project trainer saves `best_model.pt` and `latest_checkpoint.pt` on Google Drive after each epoch. `RESUME=True` continues from the last completed epoch.

In [ ]:
actual_info = None
if RUN_ACTUAL:
    actual_info = run_training(ROOT, 'actual_lie', device=DEVICE,
                               resume=RESUME, prepared=actual_prepared)
    print('Actual-lie best epoch:', actual_info['best_epoch'])
    del actual_prepared
    import gc
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## Actual-lie validation

In [ ]:
actual_val = None
if RUN_ACTUAL:
    actual_val, _ = run_evaluation(ROOT, 'actual_lie', 'validation', device=DEVICE)
    print('Actual-lie validation:', actual_val)

## Suspected-lie training

This task has its own vocabulary, model, optimizer, history and checkpoints. Unannotated receiver messages were excluded during Phase 1 processing.

In [ ]:
suspected_info = None
if RUN_SUSPECTED:
    suspected_prepared = prepare(ROOT, 'suspected_lie', device=DEVICE,
                                download_glove=DOWNLOAD_GLOVE)
    print('Suspected-lie vocabulary:', len(suspected_prepared['vocab']))
    suspected_info = run_training(ROOT, 'suspected_lie', device=DEVICE,
                                  resume=RESUME, prepared=suspected_prepared)
    print('Suspected-lie best epoch:', suspected_info['best_epoch'])
    del suspected_prepared
    import gc
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## Suspected-lie validation

In [ ]:
suspected_val = None
if RUN_SUSPECTED:
    suspected_val, _ = run_evaluation(ROOT, 'suspected_lie', 'validation', device=DEVICE)
    print('Suspected-lie validation:', suspected_val)

## Resume or evaluate without retraining

On reconnect, mount the **same** Drive project, and use `RESUME=True` to continue training. To evaluate an existing best checkpoint only, set both `RUN_*` variables to `False` and call `run_evaluation()` manually. The latest checkpoint includes optimizer and random states.

In [ ]:
for task in ['actual_lie', 'suspected_lie']:
    base = ROOT / 'artifacts' / 'models' / 'neural' / 'standard_lstm' / task
    print(task, '| best:', (base / 'best_model.pt').exists(),
          '| latest:', (base / 'latest_checkpoint.pt').exists())

## Final test evaluation

Use the untouched test set only after training and validation model selection are finished.

In [ ]:
test_results = {}
if RUN_TEST_EVALUATION:
    for task, enabled in [('actual_lie', RUN_ACTUAL), ('suspected_lie', RUN_SUSPECTED)]:
        best = ROOT / 'artifacts' / 'models' / 'neural' / 'standard_lstm' / task / 'best_model.pt'
        if best.is_file() and enabled:
            test_results[task], _ = run_evaluation(ROOT, task, 'test', device=DEVICE)
print({task: {k: r[k] for k in ['accuracy', 'macro_f1', 'lie_f1', 'pr_auc']}
       for task, r in test_results.items()})

## Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
for task in ['actual_lie', 'suspected_lie']:
    path = ROOT / 'artifacts' / 'models' / 'neural' / 'standard_lstm' / task / 'history.json'
    if path.exists():
        hist = pd.DataFrame(json.loads(path.read_text()))
        if len(hist):
            ax = hist.plot(x='epoch', y=['train_loss','validation_loss'], title=f'{task}: loss')
            fig = ax.get_figure()
            dest = ROOT / 'artifacts' / 'figures' / 'standard_lstm'
            dest.mkdir(parents=True, exist_ok=True)
            fig.savefig(dest / f'{task}_loss.png', bbox_inches='tight')
            plt.show()

## Confusion matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
for task, result in test_results.items():
    matrix = result['confusion_matrix']
    fig, ax = plt.subplots(figsize=(4,4))
    ConfusionMatrixDisplay(matrix, display_labels=['Truth', 'Lie']).plot(ax=ax, values_format='d')
    ax.set_title(task)
    dest = ROOT / 'artifacts' / 'figures' / 'standard_lstm'
    dest.mkdir(parents=True,exist_ok=True)
    fig.savefig(dest / f'{task}_confusion.png', bbox_inches='tight')
    plt.show()

## Error analysis

In [ ]:
for task in test_results:
    output = write_error_analysis(ROOT, task, 'test')
    print(task, 'errors:', output)
    display(pd.read_csv(output).head(5)[['message_id', 'target', 'prediction', 'message']])

## Compare with Phase 2

In [ ]:
for task in test_results:
    comparison = compare_phase2(ROOT, task)
    if comparison is not None:
        print(task)
        display(comparison)
    else:
        print('Phase 2 metrics unavailable for', task)

## Save parity references for local inference

Move the Drive artifacts to your local project and call `verify_parity_reference()` to check predicted labels and probabilities using the **same** checkpoint and code.

In [ ]:
for task in test_results:
    path = save_parity_reference(ROOT, task, device=DEVICE)
    print('Parity reference:', path)

## Confirm persistent storage and raw data integrity

In [ ]:
assert RAW_HASHES_BEFORE == data_hashes(), 'Original dataset was modified!'
print('Original raw files unchanged.')
for task in ['actual_lie','suspected_lie']:
    base=ROOT/'artifacts'/'models'/'neural'/'standard_lstm'/task
    print(task,'best checkpoint:',base/'best_model.pt', (base/'best_model.pt').exists())
print('All generated output is under:', ROOT/'artifacts')

## Final results summary

In [ ]:
if test_results:
    display(pd.DataFrame([
        {'task': task, 'accuracy': result['accuracy'],
         'macro_f1': result['macro_f1'], 'lie_f1': result['lie_f1'],
         'precision': result['precision'], 'recall': result['recall'],
         'pr_auc': result['pr_auc'], 'best_epoch': result['best_epoch']}
        for task, result in test_results.items()
    ]))
else:
    print('No Phase 3 test results yet. Train/evaluate a task first.')